In [6]:
from prep_regresion_familias import *

# Cargar datos
df = pd.read_pickle("data_family.pkl")
exog = df['exog'].copy().ffill()
sales_by_family = df['sales_by_family']
familias = pd.read_csv("familias.csv")

# Preparar por familia
data_by_family = {
    fam: format_as_year_month(sales_by_family[sales_by_family['family'] == fam].copy())
    for fam in sales_by_family['family'].unique()
}

# Imputar
for fam in data_by_family:
    data_by_family[fam], _ = limpiar_outliers_x_agrupacion(data_by_family[fam], 'family', 'sale_amount_MM')

# Parámetros
name = '0101'
target_col = 'sale_amount_MM'
ignore_col = [...]  # definir
negatives_reg_col = [...]  # definir

# Selección de features
feat = feature_selection(data_by_family[name], exog, max_lag=4, min_lag=-3,
                         target_col=target_col, ignore_col=ignore_col,
                         negatives_reg_col=negatives_reg_col)
feat = feat[feat['correlación'] > 0.6].reset_index(drop=True)
feat = clean_focus_correlation(feat, group='variable', focus='correlación')
feat = feat.sort_values(by='correlación', ascending=False).reset_index(drop=True)

# Reducción de colinealidad
selected, resumen = collinearity_analysis(data_by_family[name], exog, feat, target_col)
feat = feat[feat['variable'].isin(selected)]

# Dataset final + splits
df_final = construir_dataset_familia(name, data_by_family, exog, feat, target_col)
splits = generar_splits(df_final)

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/statsmodels/tsa/stattools.py:1179: RuntimeWarning: invalid value encountered in divide
  ret = cvf / (np.std(x) * np.std(y))


In [ ]:
%%capture
from optimizers import optimize_hw_cv
from optimizers import optimize_autoarima_cv
from optimizers import optimize_silverkite_cv
from optimizers import optimize_lstm_cv
from optimizers import optimize_gru_cv
from optimizers import optimize_transformer_cv
from optimizers import optimize_elastic_net_cv
from optimizers import optimize_prophet_cv
from optimizers import optimize_mlp_cv
from optimizers import optimize_transformer_cv
from optimizers import optimize_tree_cv
from optimizers import optimize_rf_cv
n_trials=5
best_params_hw, trials_df_hw, model_dict_hw = optimize_hw_cv(df_final, splits, n_trials)
trials_df_hw['model'] = 'holt_winters'
best_params_arima, trials_df_arima, model_dict_arima = optimize_autoarima_cv(df_final, splits, n_trials)
trials_df_arima['model'] = 'auto_arima'
best_params_sk, trials_df_sk, model_dict_sk = optimize_silverkite_cv(df_final, splits, n_trials)
trials_df_sk['model'] = 'silverkite'
best_params_lstm, trials_df_lstm, model_dict_lstm = optimize_lstm_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_lstm['model'] = 'lstm'
best_params_gru, trials_df_gru, model_dict_gru = optimize_gru_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_gru['model'] = 'gru'
best_params_trans, trials_df_trans, model_dict_trans = optimize_transformer_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_trans['model'] = 'transformer'
best_params_enet, trials_df_enet, model_dict_enet = optimize_elastic_net_cv(df_final.assign(ds=df_final.index),splits,n_trials)
trials_df_enet['model'] = 'elastic_net'
best_params_prophet, trials_df_prophet, model_dict_prophet = optimize_prophet_cv(df_final.reset_index(), splits,n_trials)
trials_df_prophet['model'] = 'prophet'
best_params_mlp, trials_df_mlp, model_dict_mlp = optimize_mlp_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_mlp['model'] = 'MLP'
best_params_transformer, trials_df_transformer, model_dict_transformer = optimize_transformer_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_transformer['model'] = 'Transformer'
best_params_tree, trials_df_tree, model_dict_tree = optimize_tree_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_tree['model'] = 'DecisionTree'
best_params_rf, trials_df_rf, model_dict_rf = optimize_rf_cv(df_final.assign(ds=df_final.index), splits, n_trials)
trials_df_rf['model'] = 'RandomForest'

In [35]:
import pandas as pd
import pickle
results = {}
results['trials'] = pd.concat([
    trials_df_hw.assign(model='HW'),
    trials_df_arima.assign(model='SARIMAX'),
    trials_df_sk.assign(model='Silverkite'),
    trials_df_prophet.assign(model='Prophet'),
    trials_df_mlp.assign(model='MLP'),
    trials_df_gru.assign(model='GRU'),
    trials_df_lstm.assign(model='LSTM'),
    trials_df_enet.assign(model='ElasticNet'),
    trials_df_transformer.assign(model='Transformer'),
    trials_df_tree.assign(model='RegresionTree'),
    trials_df_rf.assign(model='RandomForest'),
], ignore_index=True)

results['features']=feat
with open('results'+name+'.pkl', 'wb') as file:
    pickle.dump(results, file)

In [ ]:
import pandas as pd
df_results = pd.read_pickle("results0101.pkl")
best_models = (
    df_results['trials']
    .sort_values('mape_mean')
    .groupby('model', as_index=False)
    .first()
    .sort_values('mape_mean')
    .reset_index(drop=True)
)
best_models.to_csv("mejores_modelos_por_mape.csv", index=False)
best_models

In [87]:
params

{'n_estimators': 209,
 'max_depth': 14,
 'min_samples_split': 6,
 'min_samples_leaf': 5}

In [86]:
model, y_pred, _ = eval_func(training_set.copy(), test_set.copy(), params)

KeyError: 'seasonal_periods'

In [99]:
 funcs.items()

dict_items([('holt_winters', <function fit_predict_eval_hw at 0x14b20b6d0>)])

In [118]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_percentage_error
from models import *

# 1. Separar set de entrenamiento y prueba
df_final = df_final.assign(ds=df_final.index)
training_set = df_final.loc['2017-05-01':'2023-08-01'].copy()
test_set = df_final.loc['2023-09-01':].copy()

# 2. Filtrar el mejor MAPE promedio por modelo
models_results = df_results['trials'].reset_index(drop=True)
models_results = models_results.loc[models_results.groupby('model')['mape_mean'].idxmin()]

# 3. Definir funciones disponibles
funcs = {
    'HW': fit_predict_eval_hw,
    'SARIMAX': fit_predict_eval_autoarima,
    'Silverkite': fit_predict_eval_silverkite,
    'LSTM': fit_predict_eval_lstm,
    'GRU': fit_predict_eval_gru,
    'Transformer': fit_predict_eval_transformer,
    'ElasticNet': fit_predict_eval_elastic_net,
    'Prophet': fit_predict_eval_prophet,
    'MLP': fit_predict_eval_mlp,
    'RegresionTree': fit_predict_eval_tree,
    'RandomForest': fit_predict_eval_rf,
    # Agrega más si están disponibles
}

# 4. Iniciar estructura de resultados
predicciones = test_set[['y']].copy().rename(columns={'y': 'real'})
mape_list = []

# 5. Evaluación por modelo
for model_name, eval_func in funcs.items():
    try:
        c_model = models_results[models_results.model == model_name]
        if c_model.empty:
            raise ValueError("Modelo no encontrado en los resultados.")

        params = c_model.iloc[0]['params']
        if isinstance(params, str):
            import ast
            params = ast.literal_eval(params)

        if model_name == 'HW':
            params.setdefault('trend', 'add')
            params.setdefault('seasonal', 'add')
            params.setdefault('seasonal_periods', 12)
            params.setdefault('use_boxcox', False)

        model, y_pred, _ = eval_func(training_set.copy(), test_set.copy(), params)

        # Asegurar formato correcto
        if isinstance(y_pred, pd.Series):
            y_pred = y_pred.to_frame(name=model_name)
        elif isinstance(y_pred, pd.DataFrame):
            if y_pred.shape[1] == 1:
                y_pred.columns = [model_name]
            else:
                y_pred = y_pred.iloc[:, [0]].copy()
                y_pred.columns = [model_name]

        y_pred.index = test_set.index
        predicciones = predicciones.join(y_pred)

        mape = mean_absolute_percentage_error(test_set['y'], np.maximum(0, y_pred[model_name]))
        mape_list.append((model_name, mape))

    except Exception as e:
        print(f"⚠️ Error evaluando {model_name}: {e}")
        mape_list.append((model_name, np.nan))

# 6. Mostrar ranking final por MAPE
df_mape = pd.DataFrame(mape_list, columns=['model', 'MAPE']).sort_values(by='MAPE')
display(df_mape)

/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/pmdarima/arima/_auto_solvers.py:524: ModelFitWarning: Error fitting  ARIMA(2,1,2)(1,0,1)[6] intercept (if you do not want to see these warnings, run with error_action="ignore").
Traceback:
Traceback (most recent call last):
  File "/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/pmdarima/arima/_auto_solvers.py", line 508, in _fit_candidate_model
    fit.fit(y, X=X, **fit_params)
  File "/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/pmdarima/arima/arima.py", line 603, in fit
    self._fit(y, X, **fit_args)
  File "/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/pmdarima/arima/arima.py", line 524, in _fit
    fit, self.arima_res_ = _fit_wrapper()
  File "/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/pmdarima/arima/arima.py", line 489, in _fit_wrapper
    arima = sm.tsa.statespace.SARIMAX(
  File "/Users/sebastianulloa/.virtualen

⚠️ Error evaluando SARIMAX: Could not successfully fit a viable ARIMA model to input data.
See http://alkaline-ml.com/pmdarima/no-successful-model.html for more information on why this can happen.
⚠️ Error evaluando Silverkite: cannot insert ds, already exists


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


⚠️ Error evaluando ElasticNet: ufunc 'maximum' did not contain a loop with signature matching types (<class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.DateTime64DType'>) -> None
⚠️ Error evaluando Prophet: Prophet.__init__() got an unexpected keyword argument 'trend'


/Users/sebastianulloa/.virtualenvs/py310/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


⚠️ Error evaluando MLP: ufunc 'maximum' did not contain a loop with signature matching types (<class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.DateTime64DType'>) -> None
⚠️ Error evaluando RegresionTree: ufunc 'maximum' did not contain a loop with signature matching types (<class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.DateTime64DType'>) -> None
⚠️ Error evaluando RandomForest: ufunc 'maximum' did not contain a loop with signature matching types (<class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.DateTime64DType'>) -> None


,model,MAPE
4,GRU,0.088742
3,LSTM,0.098558
5,Transformer,0.123539
0,HW,0.127626
1,SARIMAX,NaN
2,Silverkite,NaN
6,ElasticNet,NaN
7,Prophet,NaN
8,MLP,NaN
9,RegresionTree,NaN


In [117]:
models_results

,index,trial_number,params,mape_mean,mape_best,model
36,36,1,"{'alpha': 1.8320917687360836e-05, 'l1_ratio': ...",0.069807,0.064570,ElasticNet
29,29,4,"{'window_size': 17, 'units': 68, 'epochs': 84,...",0.101182,0.079927,GRU
2,2,2,"{'trend': None, 'seasonal': 'mul', 'seasonal_p...",0.087744,0.062352,HW
30,30,0,"{'window_size': 11, 'units': 78, 'epochs': 90,...",0.101736,0.090666,LSTM
21,21,1,"{'hidden_layer_sizes': (256, 128), 'max_iter':...",0.121571,0.103246,MLP
15,15,0,"{'growth': 'flat', 'seasonality_prior_scale': ...",0.058248,0.048644,Prophet
51,51,1,"{'n_estimators': 209, 'max_depth': 14, 'min_sa...",0.050174,0.041414,RandomForest
48,48,3,"{'max_depth': 20, 'min_samples_split': 10, 'mi...",0.069376,0.060563,RegresionTree
9,9,4,"{'seasonal': True, 'm': 6, 'stepwise': True, '...",0.125243,0.067243,SARIMAX
11,11,1,"{'growth': 'sqrt', 'fit_algorithm': 'ridge', '...",0.060351,0.050562,Silverkite


In [110]:
models_results.model == model_name

36    False
29    False
2     False
30    False
21    False
15    False
51    False
48    False
9     False
11    False
44    False
Name: model, dtype: bool

In [74]:
from lag_llama.gluon.estimator import LagLlamaEstimator
from gluonts.dataset.pandas import PandasDataset
from gluonts.torch.modules.loss import NegativeLogLikelihood
from gluonts.torch.distributions.studentT import StudentTOutput
from gluonts.evaluation import make_evaluation_predictions
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd
import numpy as np
def fit_predict_eval_lagllama(training_set, test_set, model_params=None):
    if model_params is None:
        model_params = {}

    context_length = model_params.get("context_length", 12)
    prediction_length = model_params.get("prediction_length", len(test_set))
    freq = "M"

    # Garantizar que 'ds' sea solo una columna, no índice
    for df in [training_set, test_set]:
        if 'ds' in df.index.names:
            df.reset_index(inplace=True)
        if 'ds' in df.columns.duplicated():
            df = df.loc[:, ~df.columns.duplicated()]  # eliminar duplicados silenciosamente

    # Concatenar ordenadamente y asegurar unicidad de columnas
    full_df = pd.concat([training_set, test_set], axis=0)
    full_df = full_df.loc[:, ~full_df.columns.duplicated()]
    full_df = full_df.sort_values("ds").reset_index(drop=True)

    # Construcción del dataset GluonTS
    dataset = PandasDataset(
        data_frame=full_df,
        target="y",
        timestamp="ds",
        freq=freq
    )

    estimator = LagLlamaEstimator(
        prediction_length=prediction_length,
        context_length=context_length,
        input_size=1,
        loss=NegativeLogLikelihood(),
        distr_output=StudentTOutput(),
        trainer_kwargs={"max_epochs": model_params.get("epochs", 100)}
    )

    predictor = estimator.train(dataset)

    forecast_it, _ = make_evaluation_predictions(
        dataset=dataset,
        predictor=predictor,
        num_samples=100
    )

    forecasts = list(forecast_it)
    yhat = forecasts[0].mean[-prediction_length:]

    return predictor, pd.Series(yhat, index=test_set.index, name="LagLlama"), None

In [75]:
def optimize_lagllama_cv(df, split, n_trials=10):
    model_dict = {}

    def objective(trial):
        model_params = {
            "context_length": trial.suggest_int("context_length", 6, 36),
            "prediction_length": trial.suggest_int("prediction_length", 6, 24),
            "epochs": trial.suggest_int("epochs", 10, 100)
        }

        mape_scores = []
        best_mape = np.inf

        try:
            for train_index, test_index in split:
                training_set = df.iloc[train_index].copy()
                test_set = df.iloc[test_index].copy()

                _, y_pred, _ = fit_predict_eval_lagllama(training_set, test_set, model_params)
                mape = mean_absolute_percentage_error(test_set["y"].values, y_pred.values)
                mape_scores.append(mape)

                if mape < best_mape:
                    best_mape = mape

            model_dict[trial.number] = {
                "params": model_params,
                "mape_best": best_mape
            }

            return np.mean(mape_scores)

        except Exception as e:
            print(f"Error en el ensayo con parámetros {model_params}: {e}")
            return np.inf

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    trials_df = pd.DataFrame([
        {
            "trial_number": trial.number,
            "params": trial.params,
            "mape_mean": trial.value,
            "mape_best": model_dict.get(trial.number, {}).get("mape_best", np.inf)
        }
        for trial in study.trials
    ])

    return study.best_params, trials_df, model_dict

In [76]:
#

best_params_lagllama, trials_df_lagllama, model_dict_lagllama = optimize_lagllama_cv(
    df_final.assign(ds=df_final.index),  # Asegura que 'ds' esté presente
    splits,
    n_trials=10
)
trials_df_lagllama['model'] = 'lag_llama'

Error en el ensayo con parámetros {'context_length': 28, 'prediction_length': 22, 'epochs': 27}: cannot insert ds, already exists
Error en el ensayo con parámetros {'context_length': 35, 'prediction_length': 23, 'epochs': 22}: cannot insert ds, already exists
Error en el ensayo con parámetros {'context_length': 30, 'prediction_length': 17, 'epochs': 77}: cannot insert ds, already exists
Error en el ensayo con parámetros {'context_length': 11, 'prediction_length': 17, 'epochs': 55}: cannot insert ds, already exists
Error en el ensayo con parámetros {'context_length': 30, 'prediction_length': 14, 'epochs': 45}: cannot insert ds, already exists
Error en el ensayo con parámetros {'context_length': 22, 'prediction_length': 6, 'epochs': 98}: cannot insert ds, already exists
Error en el ensayo con parámetros {'context_length': 32, 'prediction_length': 13, 'epochs': 69}: cannot insert ds, already exists
Error en el ensayo con parámetros {'context_length': 34, 'prediction_length': 23, 'epochs':